# IAU 140
Hugo Foltin, Teodor Potančok

In [2]:
import pandas as pd
import seaborn as sns

Ako prvé načítame všetky dáta z CSV:

In [3]:
units = pd.read_csv("data/units.csv")
readings = pd.read_csv("data/readings.csv", sep="\t")
gateways = pd.read_csv("data/gateways.csv", sep=";")

Pre každý dataset skontrolujeme typy a počet záznamov

In [4]:
units.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   depot_id        2000 non-null   int64  
 1   code_fleet      2000 non-null   str    
 2   id_unit         2000 non-null   int64  
 3   duty_class      2000 non-null   str    
 4   service_hours   1900 non-null   float64
 5   install_region  2000 non-null   str    
 6   codeModel       2000 non-null   str    
dtypes: float64(1), int64(2), str(4)
memory usage: 109.5 KB


In [5]:
readings.info()

<class 'pandas.DataFrame'>
RangeIndex: 12313 entries, 0 to 12312
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_unit            12313 non-null  str    
 1   cycleTs            12313 non-null  str    
 2   motorCurrent       12313 non-null  float64
 3   filterDp           11782 non-null  float64
 4   oil_temperature    12313 non-null  float64
 5   reservoirPressure  12313 non-null  float64
 6   pressure_working   12313 non-null  float64
 7   oilPressure        12313 non-null  float64
 8   vibrationRms       11799 non-null  float64
 9   flowmeter          12313 non-null  float64
 10  humidityIntake     11714 non-null  float64
 11  ambient_noise      12313 non-null  float64
 12  dischargeDewpoint  12313 non-null  float64
 13  current_aux        12313 non-null  float64
 14  Failure            12313 non-null  int64  
dtypes: float64(12), int64(1), str(2)
memory usage: 1.4 MB


In [6]:
gateways.info()

<class 'pandas.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   gatewayId        16000 non-null  str    
 1   depot_id         16000 non-null  int64  
 2   installDate      16000 non-null  str    
 3   mac_address      16000 non-null  str    
 4   id_unit          2000 non-null   float64
 5   Lon              16000 non-null  float64
 6   versionFirmware  16000 non-null  str    
 7   model_hardware   16000 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 1000.1 KB


## Čistenie datasetu

Vidíme že polia v datasete majú nekonzistentné názvy - `depot_id` vs `id_unit`, `versionFirmware` vs `model_hardware`. Aby sa s dátami lepšie pracovalo musíme ich premenovať aby používali jednu konvenciu. V Pythone sa štandardne používa `camel_case` preto aj my prejdeme na tento štýl.

In [7]:
units.rename(
    columns={
        "code_fleet": "fleet_code",
        "id_unit": "unit_id",
        "codeModel": "model_code",
    },
    inplace=True,
)
units.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   depot_id        2000 non-null   int64  
 1   fleet_code      2000 non-null   str    
 2   unit_id         2000 non-null   int64  
 3   duty_class      2000 non-null   str    
 4   service_hours   1900 non-null   float64
 5   install_region  2000 non-null   str    
 6   model_code      2000 non-null   str    
dtypes: float64(1), int64(2), str(4)
memory usage: 109.5 KB


In [8]:
readings.rename(
    columns={
        "id_unit": "unit_id",
        "cycleTs": "cycle_timestamp",
        "motorCurrent": "motor_current",
        "filterDp": "filter_dp",
        "reservoirPressure": "reservoir_pressure",
        "pressure_working": "working_pressure",
        "oilPressure": "oil_pressure",
        "vibrationRms": "vibration_rms",
        "flowmeter": "flow_meter",
        "humidityIntake": "humidity_intake",
        "dischargeDewpoint": "discharge_dewpoint",
        "current_aux": "aux_current",
        "Failure": "is_failing",
    },
    inplace=True,
)
readings.info()

<class 'pandas.DataFrame'>
RangeIndex: 12313 entries, 0 to 12312
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   unit_id             12313 non-null  str    
 1   cycle_timestamp     12313 non-null  str    
 2   motor_current       12313 non-null  float64
 3   filter_dp           11782 non-null  float64
 4   oil_temperature     12313 non-null  float64
 5   reservoir_pressure  12313 non-null  float64
 6   working_pressure    12313 non-null  float64
 7   oil_pressure        12313 non-null  float64
 8   vibration_rms       11799 non-null  float64
 9   flow_meter          12313 non-null  float64
 10  humidity_intake     11714 non-null  float64
 11  ambient_noise       12313 non-null  float64
 12  discharge_dewpoint  12313 non-null  float64
 13  aux_current         12313 non-null  float64
 14  is_failing          12313 non-null  int64  
dtypes: float64(12), int64(1), str(2)
memory usage: 1.4 MB


In [9]:
gateways.rename(
    columns={
        "gatewayId": "gateway_id",
        "installDate": "install_date",
        "id_unit": "unit_id",
        "Lon": "longitude",
        "versionFirmware": "firmware_version",
        "model_hardware": "hardware_model",
    },
    inplace=True,
)
gateways.info()

<class 'pandas.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gateway_id        16000 non-null  str    
 1   depot_id          16000 non-null  int64  
 2   install_date      16000 non-null  str    
 3   mac_address       16000 non-null  str    
 4   unit_id           2000 non-null   float64
 5   longitude         16000 non-null  float64
 6   firmware_version  16000 non-null  str    
 7   hardware_model    16000 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 1000.1 KB


Mimo toho vidíme že v `gateways` je veľa záznamov v ktorých nemáme kompletné údaje - chýba `filter_dp`, `vibration_rms` alebo `humidity_intake`. Pre začiatok zistíme koľko ich je:

In [11]:
readings.isna().sum()

unit_id                 0
cycle_timestamp         0
motor_current           0
filter_dp             531
oil_temperature         0
reservoir_pressure      0
working_pressure        0
oil_pressure            0
vibration_rms         514
flow_meter              0
humidity_intake       599
ambient_noise           0
discharge_dewpoint      0
aux_current             0
is_failing              0
dtype: int64

In [13]:
readings.dropna().shape[0] / readings.shape[0] * 100

87.55786567042962

Po odstránení poškodených záznamov ich stále zostane viac ako 87%, preto sa to určite oplatí a môžeme pokračovať bez nich.

In [ ]:
readings.dropna(inplace=True)

V tabuľke `gateways` je iný problém, máme 16 000 záznamov ale len 2 000 z nich má `unit_id` ktorý potrebujeme na prepojenie s `units` a `readings`. Podľa všetkého to je zoznam sieťového vybavenie (máme MAC adresy a názvy modelov ako `NODE-9`), z ktorého väčšina nesúvisí s dopravným systémom. Záznamy bez `unit_id` sú pre nás zbytočné a je lepšie ich odstrániť.

In [14]:
gateways.dropna(inplace=True)

In [15]:
gateways.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gateway_id        2000 non-null   str    
 1   depot_id          2000 non-null   int64  
 2   install_date      2000 non-null   str    
 3   mac_address       2000 non-null   str    
 4   unit_id           2000 non-null   float64
 5   longitude         2000 non-null   float64
 6   firmware_version  2000 non-null   str    
 7   hardware_model    2000 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 125.1 KB
